# Combat PPO evaluator

Load a checkpoint, build combat from a `.jkr` snapshot (or a custom obs), and step with the policy until the episode ends. Same behavior as [`combat_evaluator.py`](combat_evaluator.py).

Run **cell 1** first (Google Colab mounts Drive and `cd`s to the repo; locally it finds the repo from the current working directory).

**Checkpoint note:** Training saves `config` as `PPOConfig`. Older runs pickled it as `__main__.PPOConfig` (from `%run BalatroPPO.ipynb`), which broke loading in this notebook. `load_combat_ppo_agent` now registers a compatible `PPOConfig` on `__main__` before `torch.load`, and `BalatroPPO.ipynb` imports the canonical class from `cs590_src.ppo_config` so new saves unpickle anywhere.

In [1]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/Duke/CS 590 RL/Project/balatro-rl-agent"
except ImportError:
    REPO_DIR = None
    for cand in Path.cwd().resolve().parents:
        if (cand / "balatro_gym").is_dir() and (cand / "cs590_env").is_dir():
            REPO_DIR = str(cand)
            break
    if REPO_DIR is None:
        cwd = Path.cwd().resolve()
        if (cwd / "balatro_gym").is_dir() and (cwd / "cs590_env").is_dir():
            REPO_DIR = str(cwd)
    if REPO_DIR is None:
        raise RuntimeError(
            "Set working directory to the repo root (containing balatro_gym and cs590_env), "
            "or run this notebook on Colab after editing REPO_DIR."
        )

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)

Mounted at /content/drive
Repo: /content/drive/MyDrive/Duke/CS 590 RL/Project/balatro-rl-agent


In [2]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
from typing import Any

import numpy as np

from balatro_gym.balatro_env_2 import BalatroEnv, DeterministicRNG
from balatro_gym.save_injection import inject_save_into_balatro_env
from cs590_env.combat_wrapper import CombatActionWrapper
from cs590_env.util import (
    combat_obs_policy_action,
    load_combat_ppo_agent,
    print_combat_state,
)
from cs590_env.wrapper import BalatroPhaseWrapper

In [3]:
_COLAB_DRIVE_REPO = "/content/drive/MyDrive/Duke/CS 590 RL/Project/balatro-rl-agent"


def _repo_root() -> Path:
    colab = Path(_COLAB_DRIVE_REPO)
    if colab.is_dir() and (colab / "balatro_gym").is_dir() and (colab / "cs590_env").is_dir():
        return colab
    cwd = Path.cwd().resolve()
    if (cwd / "balatro_gym").is_dir() and (cwd / "cs590_env").is_dir():
        return cwd
    for candidate in cwd.parents:
        if (candidate / "balatro_gym").is_dir() and (candidate / "cs590_env").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root.")


def snapshot_from_jkr(jkr_path: str | Path, *, inject_seed: int = 0) -> dict[str, Any]:
    env, _report = inject_save_into_balatro_env(Path(jkr_path), seed=inject_seed)
    return env.save_state()


def build_combat_from_snapshot(
    snapshot: dict[str, Any],
    *,
    env_seed: int,
) -> tuple[CombatActionWrapper, dict[str, np.ndarray], dict[str, Any]]:
    base = BalatroEnv()
    phase = BalatroPhaseWrapper(base)
    combat = CombatActionWrapper(phase)
    base.reset(seed=env_seed)
    base.load_state(deepcopy(snapshot))
    base.rng = DeterministicRNG(env_seed)
    phase._auto_skip_pack_open()
    obs = phase._get_phase_observation()
    obs = combat._advance_to_combat(obs)
    combat._last_obs = obs
    return combat, obs, {"env_seed": env_seed}


def evaluate_combat_episode(
    checkpoint_path: str | Path,
    *,
    jkr_path: str | Path | None = None,
    combat: CombatActionWrapper | None = None,
    initial_obs: dict[str, np.ndarray] | None = None,
    inject_seed: int = 0,
    env_seed: int = 42,
    deterministic: bool = True,
    max_steps: int = 512,
    verbose: bool = True,
    device: str | None = None,
) -> dict[str, Any]:
    if jkr_path is not None:
        snap = snapshot_from_jkr(jkr_path, inject_seed=inject_seed)
        wrap, _obs, _info = build_combat_from_snapshot(snap, env_seed=env_seed)
        if initial_obs is not None:
            wrap._last_obs = initial_obs
    else:
        if combat is None or initial_obs is None:
            raise ValueError("Without jkr_path, both combat and initial_obs are required.")
        wrap = combat
        wrap._last_obs = initial_obs
    if wrap._last_obs is None:
        raise RuntimeError("No observation to start from.")
    agent, meta = load_combat_ppo_agent(checkpoint_path, device=device)
    dev = meta["device"]
    total_reward = 0.0
    last_info: dict[str, Any] = {}
    reason = "max_steps"
    done = False
    obs: dict[str, np.ndarray] = wrap._last_obs
    step_count = 0
    for t in range(max_steps):
        if verbose:
            print(f"\n=== evaluator step {t} ===")
            print_combat_state(wrap._last_obs, env_index=0)
        card_sel, execution = combat_obs_policy_action(
            agent, wrap._last_obs, device=dev, deterministic=deterministic
        )
        if verbose:
            slots = np.flatnonzero(card_sel).tolist()
            mode = "discard" if execution else "play"
            print(f"  policy → {mode}  cards={slots}  (n={len(slots)})")
        obs, reward, done, info = wrap.step(card_sel, execution)
        step_count += 1
        total_reward += float(reward)
        last_info = dict(info) if info else {}
        if "error" in last_info:
            reason = "invalid_action"
            break
        if done:
            reason = "terminal"
            break
    if reason == "max_steps" and not done and verbose:
        print(f"\n[evaluate_combat_episode] stopped after {max_steps} steps (not terminal)")
    if done and verbose:
        print(f"\n=== evaluator final ({reason}) ===")
        print_combat_state(obs, env_index=0)
    return {
        "steps": step_count,
        "total_reward": total_reward,
        "done": done,
        "reason": reason,
        "last_info": last_info,
        "checkpoint": str(Path(checkpoint_path)),
        "device": str(dev),
        "inject_seed": inject_seed,
        "env_seed": env_seed,
    }

## Run

Adjust `CHECKPOINT`, `JKR`, and flags below. Defaults match `combat_evaluator.py` CLI.

In [4]:
root = _repo_root()
CHECKPOINT = root / "checkpoints" / "combat_ppo_iter_200.pt"
JKR = root / "save_files" / "combat" / "combat_first_round.jkr"

INJECT_SEED = 0
ENV_SEED = 42
DETERMINISTIC = True
MAX_STEPS = 512
VERBOSE = True
DEVICE = None  # e.g. "cuda" or "cpu"

assert CHECKPOINT.is_file(), f"Missing checkpoint: {CHECKPOINT}"
assert JKR.is_file(), f"Missing .jkr: {JKR}"

out = evaluate_combat_episode(
    CHECKPOINT,
    jkr_path=JKR,
    inject_seed=INJECT_SEED,
    env_seed=ENV_SEED,
    deterministic=DETERMINISTIC,
    max_steps=MAX_STEPS,
    verbose=VERBOSE,
    device=DEVICE,
)
print(
    f"\nSummary: steps={out['steps']} total_reward={out['total_reward']:.4f} "
    f"done={out['done']} reason={out['reason']!r}"
)
out

AttributeError: Can't get attribute 'PPOConfig' on <module '__main__'>